# D/D/1 - basics of queueing

This notebook reproduces the *Queueing table (1 server)* from the lecture slides: 8 customers with known (deterministic) interarrival times and service times are served by a single server.

The service station is composed of mix-ins, in the same way as objects in OpenTNSim. The customers are given as a table of interarrival and service times, so only the service station is created as an object.

The columns of the queueing table are:
* IAT: Inter-Arrival Time
* ST: Service Time
* AT: Arrival Time
* TSB: Time Service Begins
* TSE: Time Service Ends
* TCSS: Time Customer Spends in the System
* TCWQ: Time Customer Waits in the Queue
* ITS: Idle Time of Server
* QL: Queue Length (just after the customer arrives)
* SL: System Length (just after the customer arrives)

IAT and ST are the input (the *stochast* columns: in a D/D/1 queue they are known values); all other columns follow from the simulation. Times are in hours.

In [1]:
# package(s) for data handling
import pandas as pd

# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource
from openqtsim.core import create_object, inspect_object
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input
The interarrival times (IAT) and service times (ST) of the 8 customers in the queueing table:

In [ ]:
example_title = "D/D/1"

customers = pd.DataFrame({
    "IAT": [10, 5, 6, 12, 8, 7, 12, 8],  # interarrival times (hours)
    "ST": [13, 8, 4, 7, 4, 6, 7, 4],     # service times (hours)
}, index=pd.RangeIndex(1, 9, name="Customer"))
customers

,IAT,ST
Customer,,
1,10,13
2,5,8
3,6,4
4,12,7
5,8,4
6,7,6
7,12,7
8,8,4


#### 1. Create service station
Next we create a Server-object (our service station) out of the available mix-ins:

In [3]:
# make your preferred Server class out of available mix-ins.
Server = create_object(
    "Server",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of servers (c in Kendall's notation)
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [4]:
inspect_object(Server, show_parameter_table=True)

,parameter,class,default,status
0,name,Identifiable,,mandatory
1,nr_resources,HasResource,1,optional (unused)
2,id,Identifiable,None,optional (unused)


We define the required input and create the server object:

In [5]:
data_server = {
    "name": "Server",     # required by Identifiable
    "nr_resources": 1,    # optional for HasResource: one server
}

server = Server(**data_server)

#### 2. Define arrival and service processes
With the deterministic ('D') processes the customers arrive and are served exactly according to the table; arrival and service times are paired by row:

In [6]:
A = openqtsim.ArrivalProcess("D", customers[["IAT"]])
S = openqtsim.ServiceProcess("D", customers[["ST"]])

#### 3. Run simulation
The server object sets the number of servers of the queue.

In [7]:
q = openqtsim.Queue(A, S, station=server)
print('Start simulation of "{}" queue'.format(q.kendall_notation))

sim = openqtsim.Simulation(q)
sim.run(len(customers))  # each input row supplies one customer's arrival and service time

Start simulation of "D/D/1/inf/inf/FIFO" queue


In [8]:
summary = sim.get_stats()

Waiting time in units of service time: 0.3962
Observation horizon: 0 to 72.0000 hours

Rho_system: system utilisation: 0.7361
Rho_server: server utilisation: 0.7361
P_0: probability nobody in the system: 0.2639

L_s: average nr of customers in the system: 1.0278
L_q: average nr of customers in the queue: 0.2917
W_s: average time spent in the system: 9.2500
W_q: average time spent in the queue: 2.6250

IAT: average inter arrival time: 8.5000
ST: average service time: 6.6250



#### 4. Inspect output

In [9]:
df, df_sys = sim.return_log()

In [10]:
df.head(10)

# Meaning of the columns:
# c_id: customer id
# IAT: interarrival time
# ST: service time
# AT: arrival time
# TSB: time service begins
# TSE: time service ends
# TCSS: time customer spends in the system
# TCWQ: time customer waits in the queue
# ITS: idle time server
# s_id: id of server assigned to customer

,c_id,name,length,maxwait,status,departure,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,s_id
0,1,None,None,None,served,23.0,10,13,10.0,10.0,23.0,13.0,0.0,10.0,1
1,2,None,None,None,served,31.0,5,8,15.0,23.0,31.0,16.0,8.0,0.0,1
2,3,None,None,None,served,35.0,6,4,21.0,31.0,35.0,14.0,10.0,0.0,1
3,4,None,None,None,served,42.0,12,7,33.0,35.0,42.0,9.0,2.0,0.0,1
4,5,None,None,None,served,46.0,8,4,41.0,42.0,46.0,5.0,1.0,0.0,1
5,6,None,None,None,served,54.0,7,6,48.0,48.0,54.0,6.0,0.0,2.0,1
6,7,None,None,None,served,67.0,12,7,60.0,60.0,67.0,7.0,0.0,6.0,1
7,8,None,None,None,served,72.0,8,4,68.0,68.0,72.0,4.0,0.0,1.0,1


In [11]:
df_sys.head(10)

# Meaning of the columns:
# t: time of logging (if 'lambda' and 'mu' are specified as values per hour, the unit of 't' is hours)
# c_s: nr of customers in the system
# c_q nr of customers in the queue

,t,c_s,c_q
0,0.0,0,0
1,10.0,1,1
2,10.0,1,0
3,15.0,2,1
4,21.0,3,2
5,23.0,2,2
6,23.0,2,1
7,31.0,1,1
8,31.0,1,0
9,33.0,2,1


#### 5. Reproduce the queueing table
QL and SL are the queue length and system length just after a customer arrives (the arriving customer included): the customers that have arrived, but have not yet started service (QL) or not yet left (SL).

In [12]:
table = df.set_index(pd.Index(df["c_id"], name="Customer"))[
    ["IAT", "ST", "AT", "TSB", "TSE", "TCSS", "TCWQ", "ITS"]].copy()
table["QL"] = [((df["AT"] <= t) & (df["TSB"] > t)).sum() for t in df["AT"]]
table["SL"] = [((df["AT"] <= t) & (df["TSE"] > t)).sum() for t in df["AT"]]
table

,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,QL,SL
Customer,,,,,,,,,,
1,10,13,10.0,10.0,23.0,13.0,0.0,10.0,0,1
2,5,8,15.0,23.0,31.0,16.0,8.0,0.0,1,2
3,6,4,21.0,31.0,35.0,14.0,10.0,0.0,2,3
4,12,7,33.0,35.0,42.0,9.0,2.0,0.0,1,2
5,8,4,41.0,42.0,46.0,5.0,1.0,0.0,1,2
6,7,6,48.0,48.0,54.0,6.0,0.0,2.0,0,1
7,12,7,60.0,60.0,67.0,7.0,0.0,6.0,0,1
8,8,4,68.0,68.0,72.0,4.0,0.0,1.0,0,1


Check the result against the queueing table on the slide:

In [13]:
slide = pd.DataFrame({
    "IAT":  [10, 5, 6, 12, 8, 7, 12, 8],
    "ST":   [13, 8, 4, 7, 4, 6, 7, 4],
    "AT":   [10, 15, 21, 33, 41, 48, 60, 68],
    "TSB":  [10, 23, 31, 35, 42, 48, 60, 68],
    "TSE":  [23, 31, 35, 42, 46, 54, 67, 72],
    "TCSS": [13, 16, 14, 9, 5, 6, 7, 4],
    "TCWQ": [0, 8, 10, 2, 1, 0, 0, 0],
    "ITS":  [10, 0, 0, 0, 0, 2, 6, 1],
    "QL":   [0, 1, 2, 1, 1, 0, 0, 0],
    "SL":   [1, 2, 3, 2, 2, 1, 1, 1],
}, index=pd.RangeIndex(1, 9, name="Customer"))

pd.testing.assert_frame_equal(table, slide, check_dtype=False)
print("The simulation reproduces the queueing table of the slide.")

The simulation reproduces the queueing table of the slide.


#### 6. Performance measures: W<sub>s</sub>, W<sub>q</sub>, L<sub>q</sub> and L<sub>s</sub>
W<sub>s</sub> and W<sub>q</sub> are the averages of the TCSS and TCWQ columns. For L<sub>q</sub> and L<sub>s</sub> there are two averages:
* the average of the QL and SL columns: the queue and system length **as seen by the arriving customers**;
* the **time average** reported by `sim.get_stats()`: the number of customers in the queue or system, averaged over the observation period (0 to the last departure, 72 hours). By Little's law this equals L = λ·W, with λ = 8 customers / 72 hours.

For Poisson arrivals both averages are the same in the long run (PASTA: Poisson Arrivals See Time Averages). With these deterministic arrivals they differ: the queue lengths seen by arriving customers are larger than the time averages.

In [ ]:
measures = pd.DataFrame({
    "average over arriving customers": {
        "W_s": table["TCSS"].mean(), "W_q": table["TCWQ"].mean(),
        "L_q": table["QL"].mean(), "L_s": table["SL"].mean()},
    "time average (sim.get_stats)": {
        "W_s": summary["W_s"], "W_q": summary["W_q"], "L_q": summary["L_q"], "L_s": summary["L_s"]},
})
measures.round(4) 

,average over arriving customers,time average (sim.get_stats)
W_s,9.250,9.2500
W_q,2.625,2.6250
L_q,0.625,0.2917
L_s,1.625,1.0278


#### 7. Plot system state, interarrival times and service times

In [15]:
sim.plot_system_state()

In [16]:
sim.plot_IAT_ST()

#### 8. Gantt chart: arrival, waiting, service and departure

Each row is one customer, in arrival order. Orange spans arrival (`AT`) to service start (`TSB`); blue spans service start to departure (`TSE`). Hover to inspect the times in hours (and days) since simulation start.

In [17]:
fig = plot_gantt(df, title=f"{example_title}: queueing table (1 server, {len(df)} customers)", height=450)
fig.show()